In [2]:
from dataset_generator.dataset import CrossChainTransactionsDataset
import torch
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.utils import to_undirected

/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load the dataset
dataset = CrossChainTransactionsDataset(root='out/', force_reload=True)
dataset[0]

HeteroData(
  y=[1],
  y_str='normal',
  user={ x=[2, 9] },
  router={ x=[2, 9] },
  log_event={ x=[6, 25] },
  token={ x=[2, 17] },
  validator={ x=[1, 12] },
  (user, token_auth, router)={ edge_index=[2, 1] },
  (token, log_relation, log_event)={ edge_index=[2, 3] },
  (user, token_transfer, router)={ edge_index=[2, 1] },
  (user, transaction, router)={ edge_index=[2, 1] },
  (router, function_call, token)={ edge_index=[2, 2] },
  (router, log_relation, log_event)={ edge_index=[2, 3] },
  (router, token_transfer, user)={ edge_index=[2, 1] },
  (router, cross_chain_relation, validator)={ edge_index=[2, 1] },
  (validator, cross_chain_relation, router)={ edge_index=[2, 1] }
)

In [ ]:
# Check for NaN or infinite values in node and edge features
nan_inf_indices = []
for idx, data in enumerate(dataset):
    has_nan_or_inf = False
    
    # Check node features
    for node_type in data.node_types:
        if hasattr(data[node_type], 'x') and data[node_type].x is not None:
            if torch.isnan(data[node_type].x).any() or torch.isinf(data[node_type].x).any():
                has_nan_or_inf = True
                break
    
    # Check edge features
    if not has_nan_or_inf:
        for edge_type in data.edge_types:
            if hasattr(data[edge_type], 'edge_attr') and data[edge_type].edge_attr is not None:
                if torch.isnan(data[edge_type].edge_attr).any() or torch.isinf(data[edge_type].edge_attr).any():
                    has_nan_or_inf = True
                    break
    
    if has_nan_or_inf:
        nan_inf_indices.append(idx)

print(f"Graphs with NaN or Inf values at indices: {nan_inf_indices}")

Graphs with NaN values at indices: []


In [ ]:
# Get the sum of each label in the dataset. Each label is a one-hot encoded vector
label_counts = torch.zeros(4, dtype=torch.long)
for data in dataset:
    label_counts += data.y

label_counts

tensor([292,   0,   0,   0])

In [1]:
from dataset_generator.model_training import train
train('./out', {})

/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Identified 36 differential meta-paths for training.
Start training on fold 1/5...


ValueError: too many fds

In [ ]:
dataset[0].edge_types

[('user', 'token_auth', 'router'),
 ('token', 'log_relation', 'log_event'),
 ('user', 'token_transfer', 'router'),
 ('user', 'transaction', 'router'),
 ('router', 'function_call', 'token'),
 ('router', 'log_relation', 'log_event'),
 ('router', 'token_transfer', 'user'),
 ('router', 'cross_chain_relation', 'validator'),
 ('validator', 'cross_chain_relation', 'router')]

In [ ]:
# Neighbor aggregation example
data = dataset[0]
num_hops = 1

# Create k-hop metapath features via row-normalized adjacency products (SeHGNN-style).
def aggregate_neighbors(data, num_hops):
    aggregated_features = {}

    # Base case: raw features for each node type, keyed as 0-hop metapath tuples.
    for node_type in data.node_types:
        if not hasattr(data[node_type], 'x') or data[node_type].x is None:
            raise ValueError(f"Node type '{node_type}' has no feature matrix 'x'.")
        aggregated_features[(node_type,)] = data[node_type].x

    if num_hops == 0:
        return aggregated_features

    # Build merged adjacency matrices A(src, dst) with shape [num_src, num_dst].
    # If multiple relation types share the same (src, dst), merge first, normalize once after merging.
    #* NOTE: SeHGNN does not explicitly tell how neighbor aggregation should be
    #* handled when we're dealing with directional edges. This affects how the
    #* adjacency matrices are constructed and normalized, and ultimately the
    #* features we get after aggregation.
    #* After analyzing tradeoffs, we decide to preserve edge directionality when
    #* computing the adjacency matrices, as this allows us to capture a causal
    #* meaning in the graph structure, which is especially important when dealing
    #* with token transfer flows, for example.
    raw_adj_by_type_pair = {}
    for src_type, _, dst_type in data.edge_types:
        edge_index = data[(src_type, _, dst_type)].edge_index
        num_src = data[src_type].num_nodes
        num_dst = data[dst_type].num_nodes

        values = torch.ones(edge_index.size(1), dtype=torch.float32, device=edge_index.device)
        a = torch.sparse_coo_tensor(
            edge_index,
            values,
            size=(num_src, num_dst),
            device=edge_index.device,
        ).coalesce()

        key = (src_type, dst_type)
        if key in raw_adj_by_type_pair:
            raw_adj_by_type_pair[key] = (raw_adj_by_type_pair[key] + a).coalesce()
        else:
            raw_adj_by_type_pair[key] = a

    # Row-normalize merged adjacency per (src, dst): A_hat = D^{-1} A
    adj_by_type_pair = {}
    for key, a in raw_adj_by_type_pair.items():
        row_sum = torch.sparse.sum(a, dim=1).to_dense()
        inv_row_sum = torch.where(row_sum > 0, 1.0 / row_sum, torch.zeros_like(row_sum))
        norm_values = a.values() * inv_row_sum[a.indices()[0]]
        a_hat = torch.sparse_coo_tensor(
            a.indices(),
            norm_values,
            size=a.size(),
            device=a.device,
        ).coalesce()
        adj_by_type_pair[key] = a_hat

    # Iteratively build longer metapaths by prepending one normalized adjacency.
    # If X^{path} is known and path starts with 'mid', then for edge src->mid:
    # X^{(src,)+path} = A_hat(src, mid) @ X^{path}
    current_level = dict(aggregated_features)
    for i in range(1, num_hops + 1):
        next_level = {}
        for path, path_features in current_level.items():
            mid_type = path[0]
            for (src_type, dst_type), a_hat in adj_by_type_pair.items():
                if dst_type != mid_type:
                    continue
                new_path = (src_type,) + path
                next_level[new_path] = torch.sparse.mm(a_hat, path_features)

        aggregated_features.update(next_level)
        current_level = next_level

    return aggregated_features

In [ ]:
test_data = HeteroData()
test_data['p'].x = torch.tensor([[1, 2], [3, 4], [5, 6]]).float()
test_data['a'].x = torch.tensor([[10, 20], [30, 40]]).float()
test_data['s'].x = torch.tensor([[100, 200]]).float()

test_data[('p', 'w', 'a')].edge_index = torch.tensor([[0, 1, 2], [0, 0, 1]]).float()
test_data[('p', 's', 's')].edge_index = torch.tensor([[0, 2], [0, 0]]).float()

aggregate_neighbors(test_data, 2)

{('p',): tensor([[1., 2.],
         [3., 4.],
         [5., 6.]]),
 ('a',): tensor([[10., 20.],
         [30., 40.]]),
 ('s',): tensor([[100., 200.]]),
 ('a',
  'p'): tensor([[2., 3.],
         [5., 6.]]),
 ('s', 'p'): tensor([[3., 4.]]),
 ('p',
  'a'): tensor([[10., 20.],
         [10., 20.],
         [30., 40.]]),
 ('p',
  's'): tensor([[100., 200.],
         [  0.,   0.],
         [100., 200.]]),
 ('p',
  'a',
  'p'): tensor([[2., 3.],
         [2., 3.],
         [5., 6.]]),
 ('p',
  's',
  'p'): tensor([[3., 4.],
         [0., 0.],
         [3., 4.]]),
 ('a',
  'p',
  'a'): tensor([[10., 20.],
         [30., 40.]]),
 ('s', 'p', 'a'): tensor([[20., 30.]]),
 ('a',
  'p',
  's'): tensor([[ 50., 100.],
         [100., 200.]]),
 ('s', 'p', 's'): tensor([[100., 200.]])}

In [ ]:
aggregate_neighbors(data, 4)

{('user',): tensor([[0., 3., 1., 0., 0., 0., 0., 0., 1.],
         [1., 0., 0., 1., 0., 1., 1., 0., 0.]]),
 ('router',): tensor([[1., 4., 0., 1., 0., 1., 1., 0., 0.],
         [3., 3., 1., 0., 0., 0., 0., 0., 1.]]),
 ('token',): tensor([[1.0000, 2.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
          0.0000, 0.2667, 0.3333, 0.4000, 0.9333, 0.5333, 0.6667, 0.6000],
         [1.0000, 1.0000, 0.0000, 1.0000, 0.0000, 1.0000, 1.0000, 0.0000, 0.0000,
          0.0000, 0.2667, 0.3333, 0.4000, 0.9333, 0.5333, 0.6667, 0.6000]]),
 ('log_event',): tensor([[ 1.0000,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           1.0000,  0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  3.0000,  2.0000,
           0.0000,  0.0000,  0.2667,  0.3333,  0.4000,  0.9333,  0.5333,  0.6667,
           0.6000],
         [ 1.0000,  0.0000,  1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           1.0000,  0.5000,  0.0000,  0.0000,  0.0000,  1.0000,  3.0000,  2.0000,
          